In [6]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

bike_parking = RAW_DIR / 'bike_parking.geojson'
boundaries = RAW_DIR / 'boundaries.geojson'

bike_parking = gpd.read_file(bike_parking)
boundaries = gpd.read_file(boundaries)

print('bike parking', bike_parking.shape)
print('boundaries', boundaries.shape)
print('bike parking CRS:', bike_parking.crs)
print('boundaries CRS:', boundaries.crs)

bike parking (3127, 58)
boundaries (264, 36)
bike parking CRS: EPSG:4326
boundaries CRS: EPSG:4326


In [7]:
boundaries.columns.to_list()
boundaries[[col for col in boundaries.columns if 'name' in col.lower()]].head()

,alt_name,loc_name,name,name:be,name:ca,name:de,name:en,name:eo,name:etymology:wikidata,name:it,name:pt,name:ru,name:uk,name:vi,name:zh,name:zh-Hans,name:zh-Hant
0,муниципальный округ Кунцево,NaN,район Кунцево,NaN,Districte de Kúntsevo,NaN,Kuntsevo District,NaN,NaN,NaN,NaN,район Кунцево,Район Кунцеве,NaN,NaN,NaN,NaN
1,муниципальный округ Крылатское,NaN,район Крылатское,Крылацкае,NaN,NaN,Krylatskoye District,NaN,NaN,NaN,NaN,район Крылатское,Район Крилатське,NaN,NaN,NaN,NaN
2,муниципальный округ Мещанский,NaN,Мещанский район,NaN,NaN,NaN,Meshchansky District,NaN,NaN,NaN,NaN,Мещанский район,Міщанський район,NaN,NaN,NaN,NaN
3,муниципальный округ Сокольники,NaN,район Сокольники,NaN,NaN,NaN,Sokolniki District,NaN,NaN,NaN,NaN,район Сокольники,Район Сокільники,NaN,NaN,NaN,NaN
4,NaN,NaN,район Коммунарка,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
targer_crs = 'EPSG:32637'

bike_parking = bike_parking.to_crs(targer_crs)
boundaries = boundaries.to_crs(targer_crs)

print('bike parking CRS:', bike_parking.crs)
print('boundaries CRS:', boundaries.crs)

bike parking CRS: EPSG:32637
boundaries CRS: EPSG:32637


In [9]:
bike_parking = bike_parking[bike_parking.geometry.notna()].copy()
boundaries = boundaries[boundaries.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()

print('bike parking valid:', bike_parking.shape)
print('boundaries valid:', boundaries.shape)

bike parking valid: (3127, 58)
boundaries valid: (132, 36)


In [10]:
bike_parking_with_bound = gpd.sjoin(bike_parking, boundaries[['name', 'geometry']], how = 'left', predicate = 'within', rsuffix='bound') 
bike_parking_with_bound.head()

,id,@id,access,addr:floor,alt_name,amenity,bench,bicycle_parking,bike_ride,bin,...,surface,surveillance,survey:date,tactile_paving,type,website,@geometry,geometry,index_bound,name_bound
0,relation/16283376,relation/16283376,private,NaN,NaN,bicycle_parking,NaN,shed,NaN,NaN,...,NaN,NaN,NaT,NaN,multipolygon,NaN,center,POINT (412557.372 6175697.772),13,Донской район
1,way/165051502,way/165051502,yes,NaN,NaN,bicycle_parking,NaN,wall_loops,NaN,NaN,...,NaN,no,NaT,NaN,NaN,NaN,center,POINT (388226.399 6206946.569),124,район Матушкино
2,way/258553674,way/258553674,NaN,NaN,NaN,bicycle_parking,NaN,wall_loops,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,center,POINT (407685.312 6173851.197),43,район Раменки
3,way/298268772,way/298268772,NaN,NaN,NaN,bicycle_parking,NaN,rack,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,center,POINT (422598.219 6188270.262),117,район Метрогородок
4,way/298268773,way/298268773,NaN,NaN,NaN,bicycle_parking,NaN,rack,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,center,POINT (425277.79 6187933.156),118,район Гольяново


In [11]:
bike_parking_with_bound.geometry.isna().sum()

np.int64(0)

In [12]:
bike_parking_counts = (bike_parking_with_bound.
    groupby('name_bound').
    size().
    reset_index(name='bike_parking_count'))

bike_parking_counts.head()

,name_bound,bike_parking_count
0,Академический район,31
1,Алексеевский район,9
2,Алтуфьевский район,1
3,Бабушкинский район,23
4,Басманный район,168


In [13]:
boundaries_stats = boundaries.merge(bike_parking_counts, left_on = 'name', right_on='name_bound', how='left')
boundaries_stats['bike_parking_count'] = boundaries_stats['bike_parking_count'].fillna(0).astype(int)
boundaries_stats.head()

,id,@id,@relations,addr:country,addr:region,admin_level,alt_name,boundary,contact:website,description,...,omkmo:code,omkte:code,population,type,website,wikidata,wikipedia,geometry,name_bound,bike_parking_count
0,relation/226927,relation/226927,NaN,RU,Москва,8,муниципальный округ Кунцево,administrative,NaN,район — административно-территориальная единиц...,...,NaN,NaN,NaN,boundary,NaN,Q155102,ru:Кунцево (район Москвы),"MULTIPOLYGON (((394009.876 6184651.829, 394033...",район Кунцево,13
1,relation/240229,relation/240229,NaN,RU,Москва,8,муниципальный округ Крылатское,administrative,NaN,район — административно-территориальная единиц...,...,42,806,NaN,boundary,NaN,Q474850,ru:Крылатское (район),"POLYGON ((400976.618 6178845.806, 401034.953 6...",район Крылатское,7
2,relation/364001,relation/364001,NaN,NaN,NaN,8,муниципальный округ Мещанский,administrative,NaN,район — административно-территориальная единиц...,...,58,105,NaN,boundary,NaN,Q2626320,ru:Мещанский район,"POLYGON ((413490.877 6180149.073, 413491.462 6...",Мещанский район,40
3,relation/364551,relation/364551,NaN,RU,NaN,8,муниципальный округ Сокольники,administrative,NaN,район — административно-территориальная единиц...,...,097,0418,NaN,boundary,NaN,Q1935566,ru:Сокольники (район Москвы),"POLYGON ((416614.177 6186996.283, 416562.301 6...",район Сокольники,37
4,relation/380704,relation/380704,NaN,NaN,NaN,8,NaN,administrative,NaN,NaN,...,NaN,NaN,NaN,boundary,NaN,NaN,NaN,"POLYGON ((406516.111 6157030.578, 406649.653 6...",район Коммунарка,124


In [14]:
non_zero = boundaries_stats.loc[boundaries_stats['bike_parking_count'] > 0, 'bike_parking_count']
q1 = non_zero.quantile(1/3)
q2 = non_zero.quantile(2/3)

def classify_parkin(count):
    if count == 0:
        return 'нет'
    elif count < q1:
        return 'мало'
    elif q1 < count < q2:
        return 'средне'
    else:
        return 'много'

boundaries_stats['bike_parking_zone'] = boundaries_stats['bike_parking_count'].apply(classify_parkin)
boundaries_stats[['name_bound', 'bike_parking_count', 'bike_parking_zone']].sort_values('bike_parking_count')

,name_bound,bike_parking_count,bike_parking_zone
129,NaN,0,нет
130,NaN,0,нет
45,NaN,0,нет
126,район Бекасово,1,мало
66,район Бирюлёво Западное,1,мало
...,...,...,...
95,Пресненский район,109,много
4,район Коммунарка,124,много
88,район Хамовники,127,много
94,Тверской район,143,много


In [15]:
output_path = PROCESSED_DIR / 'bound_bike_parking.gpkg'

boundaries_stats.to_file(
    output_path,
    layer='bound_bike_parking',
    driver='GPKG'
)

print(f'Save to: {output_path}')

Save to: /Users/liza/Desktop/project/data/processed/bound_bike_parking.gpkg
